# NSYS 2026: HPO for Physics-Informed Neural Networks (REAL PyTorch Training)Self-contained Colab notebook — **real PyTorch PINN training** (autograd physicsresiduals, real MLP models, real optimizers), ported verbatim from this repo's`src/training/benchmark_factory.py`, `src/models/mlp.py`, and `src/benchmarks/*`.No synthetic/placeholder metrics for ODE, Heat, Burgers, or Wave.Benchmarks comparing 6 metaheuristics:- **GA**, **PSO**, **ACO**- **Fuzzy-GA**, **Fuzzy-PSO**, **Fuzzy-ACO** (Mamdani fuzzy controller adapts search parameters)**Task Breakdown:** 6 algorithms × 4 benchmarks (ODE, Heat, Burgers, Wave) × 3 seeds= **72 real optimization tasks**, plus 5 reporting tasks = **77 total**.**IMPORTANT — set `QUICK_MODE = True` first** to smoke-test the whole pipeline in afew minutes before committing to the full multi-hour run (Cell "Configuration" below).**Estimated full-run time:** several hours on a T4 GPU (real gradient descent on ~90-110PINN training calls per algorithm/benchmark/seed cell, 1200 steps each). Enable GPU:**Runtime → Change runtime type → T4 GPU**.

In [ ]:
import subprocess, syspackages = ["torch", "numpy", "scipy", "matplotlib", "tqdm"]for pkg in packages:    try:        __import__(pkg)        print(f"already installed: {pkg}")    except ImportError:        print(f"installing: {pkg}")        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])print("\nAll dependencies ready.")

In [ ]:
import os, json, timefrom pathlib import Pathfrom dataclasses import dataclass, asdict, replacefrom typing import Anyfrom functools import lru_cacheimport warningsimport numpy as npimport matplotlib.pyplot as pltimport torchimport torch.nn as nnfrom tqdm.notebook import tqdmwarnings.filterwarnings("ignore")OUTPUT_DIR = "/content/nsys2026_results"Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)DEVICE = "cuda" if torch.cuda.is_available() else "cpu"print(f"Using device: {DEVICE}")if DEVICE == "cpu":    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU (T4) is strongly recommended.")print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
def set_seed(seed: int):    np.random.seed(seed)    torch.manual_seed(seed)    if torch.cuda.is_available():        torch.cuda.manual_seed_all(seed)def ensure_dir(path: str):    Path(path).mkdir(parents=True, exist_ok=True)def save_json(path: str, data: dict):    ensure_dir(str(Path(path).parent))    with open(path, 'w') as f:        json.dump(data, f, indent=2, default=str)

In [ ]:
@dataclass(frozen=True)class SearchSpace:    hidden_layers_min: int = 1    hidden_layers_max: int = 6    hidden_width_min: int = 8    hidden_width_max: int = 256    activations: tuple = ("tanh", "sine", "swish")    optimizers: tuple = ("adam", "adamw", "lbfgs")    lr_min: float = 1e-4    lr_max: float = 5e-2    w_phys_min: float = 0.1    w_phys_max: float = 10.0    w_ic_min: float = 0.1    w_ic_max: float = 50.0    n_collocation_min: int = 64    n_collocation_max: int = 1024    def get_bounds(self):        lb = np.array([            float(self.hidden_layers_min), float(self.hidden_width_min), 0.0, 0.0,            float(np.log10(self.lr_min)), float(self.w_phys_min),            float(self.w_ic_min), float(self.n_collocation_min)        ], dtype=float)        ub = np.array([            float(self.hidden_layers_max), float(self.hidden_width_max),            float(len(self.activations) - 1), float(len(self.optimizers) - 1),            float(np.log10(self.lr_max)), float(self.w_phys_max),            float(self.w_ic_max), float(self.n_collocation_max)        ], dtype=float)        return lb, ubdef clip_int(x, lo, hi):    return max(lo, min(hi, int(round(float(x)))))def clip_float(x, lo, hi):    return max(lo, min(hi, float(x)))def choose_activation(idx, activations):    i = max(0, min(len(activations) - 1, int(round(float(idx)))))    return activations[i]def choose_optimizer(idx, optimizers):    i = max(0, min(len(optimizers) - 1, int(round(float(idx)))))    return optimizers[i]def decode_solution(x: np.ndarray, space: SearchSpace, base) -> "TrainConfig":    layers = clip_int(x[0], space.hidden_layers_min, space.hidden_layers_max)    width = clip_int(x[1], space.hidden_width_min, space.hidden_width_max)    activation = choose_activation(x[2], space.activations)    optimizer = choose_optimizer(x[3], space.optimizers)    log10_lr = clip_float(x[4], np.log10(space.lr_min), np.log10(space.lr_max))    lr = float(10 ** log10_lr)    w_phys = clip_float(x[5], space.w_phys_min, space.w_phys_max)    w_ic = clip_float(x[6], space.w_ic_min, space.w_ic_max)    n_col = clip_int(x[7], space.n_collocation_min, space.n_collocation_max)    return replace(base, hidden_layers=layers, hidden_width=width, activation=activation,                   optimizer=optimizer, lr=lr, w_phys=w_phys, w_ic=w_ic, n_collocation=n_col)

In [ ]:
def _trimf(x, abc):    a, b, c = abc    x_arr = np.asarray(x, dtype=float)    y = np.zeros_like(x_arr)    if b != a:        left_mask = (a <= x_arr) & (x_arr <= b)        y[left_mask] = (x_arr[left_mask] - a) / (b - a)    if c != b:        right_mask = (b <= x_arr) & (x_arr <= c)        y[right_mask] = (c - x_arr[right_mask]) / (c - b)    y[x_arr == b] = 1.0    y = np.clip(y, 0.0, 1.0)    return float(y) if np.isscalar(x) else yclass FuzzyController:    def __init__(self):        self.div_low = (0.0, 0.0, 0.45)        self.div_med = (0.2, 0.5, 0.8)        self.div_high = (0.55, 1.0, 1.0)        self.imp_stagnant = (0.0, 0.0, 0.3)        self.imp_slow = (0.15, 0.5, 0.85)        self.imp_fast = (0.6, 1.0, 1.0)        self.prog_early = (0.0, 0.0, 0.45)        self.prog_mid = (0.25, 0.5, 0.75)        self.prog_late = (0.55, 1.0, 1.0)        self.u_out = np.linspace(0.0, 1.0, 101)        self.out_low = _trimf(self.u_out, (0.0, 0.0, 0.5))        self.out_med = _trimf(self.u_out, (0.25, 0.5, 0.75))        self.out_high = _trimf(self.u_out, (0.5, 1.0, 1.0))    def evaluate(self, diversity, improvement_rate, iteration_progress):        d = float(np.clip(diversity, 0.0, 1.0))        imp = float(np.clip(improvement_rate, 0.0, 1.0))        prog = float(np.clip(iteration_progress, 0.0, 1.0))        mu_d_low = float(_trimf(d, self.div_low)); mu_d_med = float(_trimf(d, self.div_med)); mu_d_high = float(_trimf(d, self.div_high))        mu_imp_stag = float(_trimf(imp, self.imp_stagnant)); mu_imp_slow = float(_trimf(imp, self.imp_slow)); mu_imp_fast = float(_trimf(imp, self.imp_fast))        mu_prog_early = float(_trimf(prog, self.prog_early)); mu_prog_mid = float(_trimf(prog, self.prog_mid)); mu_prog_late = float(_trimf(prog, self.prog_late))        r1 = min(mu_d_low, mu_imp_stag)        r2 = min(mu_d_high, mu_imp_fast)        r3 = mu_prog_early        r4 = min(mu_prog_late, mu_imp_stag)        r5 = min(mu_prog_late, mu_imp_fast)        r6 = min(mu_d_med, mu_imp_slow)        r7 = mu_prog_mid        exp_high = max(r1, r3); exp_med = max(r4, r6, r7); exp_low = max(r2, r5)        agg_exp = np.maximum(np.minimum(exp_high, self.out_high),                    np.maximum(np.minimum(exp_med, self.out_med), np.minimum(exp_low, self.out_low)))        expt_high = max(r2, r4, r5); expt_med = max(r3, r6, r7); expt_low = r1        agg_expt = np.maximum(np.minimum(expt_high, self.out_high),                    np.maximum(np.minimum(expt_med, self.out_med), np.minimum(expt_low, self.out_low)))        sum_exp = np.sum(agg_exp)        exploration = float(np.sum(self.u_out * agg_exp) / sum_exp) if sum_exp > 1e-9 else 0.5        sum_expt = np.sum(agg_expt)        exploitation = float(np.sum(self.u_out * agg_expt) / sum_expt) if sum_expt > 1e-9 else 0.5        return exploration, exploitationdef compute_population_diversity(population, lb, ub):    if len(population) <= 1:        return 0.0    norm_pop = (population - lb) / (ub - lb + 1e-12)    centroid = np.mean(norm_pop, axis=0)    distances = np.linalg.norm(norm_pop - centroid, axis=1)    max_d = np.sqrt(norm_pop.shape[1]) * 0.5    div = float(np.mean(distances) / (max_d + 1e-12))    return float(np.clip(div, 0.0, 1.0))

In [ ]:
# ---- Real MLP model (verbatim port of src/models/mlp.py) ----class Sine(nn.Module):    def forward(self, x):        return torch.sin(x)def _activation(name: str) -> nn.Module:    name = name.lower()    if name == "tanh": return nn.Tanh()    if name in {"silu", "swish"}: return nn.SiLU()    if name == "relu": return nn.ReLU()    if name == "sine": return Sine()    raise ValueError(f"Unknown activation: {name}")class MLP(nn.Module):    def __init__(self, in_dim, out_dim, hidden_layers, hidden_width, activation="tanh"):        super().__init__()        act = _activation(activation)        layers = []        last = in_dim        for _ in range(int(hidden_layers)):            layers.append(nn.Linear(last, int(hidden_width)))            layers.append(act)            last = int(hidden_width)        layers.append(nn.Linear(last, out_dim))        self.net = nn.Sequential(*layers)    def forward(self, x):        return self.net(x)# ---- Real benchmark classes (verbatim port of src/benchmarks/*) ----@dataclass(frozen=True)class ExponentialDecayBenchmark:    """ODE: y'(t) = -y(t), y(t0)=y0. Analytic: y(t)=y0*exp(-(t-t0))."""    t0: float = 0.0    t1: float = 5.0    y0: float = 1.0    def y_true(self, t):        t = np.asarray(t, dtype=float)        return self.y0 * np.exp(-(t - self.t0))    def residual(self, t, y, dy_dt):        return dy_dt + y@dataclass(frozen=True)class HeatEquationBenchmark:    """1D Heat: u_t = alpha*u_xx. Analytic: sin(pi*x/L)*exp(-alpha*pi^2*t/L^2)."""    x0: float = 0.0    x1: float = 1.0    t1: float = 1.0    alpha: float = 0.1    def domain(self):        return ((self.x0, self.x1), (0.0, self.t1))    def initial_condition(self, x):        L = self.x1 - self.x0        return np.sin(np.pi * (x - self.x0) / L)    def boundary_conditions(self, t):        return np.zeros_like(t), np.zeros_like(t)    def analytic_solution(self, x, t):        L = self.x1 - self.x0        return np.sin(np.pi * (x - self.x0) / L) * np.exp(-self.alpha * np.pi**2 * t / L**2)    def residual(self, x, t, u, u_t, u_xx):        return u_t - self.alpha * u_xx@dataclass(frozen=True)class Burgers1DBenchmark:    """1D viscous Burgers: u_t + u*u_x = nu*u_xx. No closed form -> FD reference."""    x0: float = -1.0    x1: float = 1.0    t1: float = 1.0    nu: float = 0.01    def domain(self):        return ((self.x0, self.x1), (0.0, self.t1))    def initial_condition(self, x):        return np.sin(np.pi * x)    def boundary_conditions(self, t):        return np.zeros_like(t), np.zeros_like(t)    def residual(self, x, t, u, u_t, u_x, u_xx):        return u_t + u * u_x - self.nu * u_xx@dataclass(frozen=True)class WaveEquationBenchmark:    """1D Wave: u_tt = c^2*u_xx. Analytic: sin(pi*x/L)*cos(c*pi*t/L)."""    x0: float = 0.0    x1: float = 1.0    t1: float = 2.0    c: float = 1.0    def domain(self):        return ((self.x0, self.x1), (0.0, self.t1))    def initial_condition_u(self, x):        L = self.x1 - self.x0        return np.sin(np.pi * (x - self.x0) / L)    def initial_condition_u_t(self, x):        return np.zeros_like(x)    def boundary_conditions(self, t):        return np.zeros_like(t), np.zeros_like(t)    def analytic_solution(self, x, t):        L = self.x1 - self.x0        return np.sin(np.pi * (x - self.x0) / L) * np.cos(self.c * np.pi * t / L)    def residual(self, x, t, u, u_tt, u_xx):        return u_tt - self.c**2 * u_xxdef get_benchmark(benchmark_type: str):    return {        "ode": ExponentialDecayBenchmark(),        "heat": HeatEquationBenchmark(),        "burgers": Burgers1DBenchmark(),        "wave": WaveEquationBenchmark(),    }[benchmark_type]print("Real MLP model and benchmark classes loaded.")

In [ ]:
@dataclass(frozen=True)class TrainConfig:    seed: int = 0    device: str = DEVICE    benchmark_type: str = "ode"    t0: float = 0.0    t1: float = 5.0    n_eval: int = 200    hidden_layers: int = 3    hidden_width: int = 32    activation: str = "tanh"    optimizer: str = "adam"    lbfgs_max_iter: int = 3    lr: float = 1e-3    n_steps: int = 1200    n_collocation: int = 256    w_phys: float = 1.0    w_ic: float = 10.0def _make_optimizer(model, cfg):    opt_name = cfg.optimizer.lower()    if opt_name == "adam":        return torch.optim.Adam(model.parameters(), lr=float(cfg.lr)), False    elif opt_name == "adamw":        return torch.optim.AdamW(model.parameters(), lr=float(cfg.lr)), False    elif opt_name == "lbfgs":        return torch.optim.LBFGS(model.parameters(), lr=float(cfg.lr), max_iter=int(cfg.lbfgs_max_iter)), True    else:        raise ValueError(f"Unsupported optimizer: {cfg.optimizer}")def _run_optimization(opt, use_lbfgs, compute_loss_fn, n_steps):    last_loss = None    if use_lbfgs:        for _ in range(int(n_steps)):            def closure():                opt.zero_grad()                loss = compute_loss_fn()                loss.backward()                return loss            loss_tensor = opt.step(closure)            last_loss = float(loss_tensor.detach().cpu().item())    else:        for _ in range(int(n_steps)):            opt.zero_grad(set_to_none=True)            loss = compute_loss_fn()            loss.backward()            opt.step()            last_loss = float(loss.detach().cpu().item())    return last_lossdef _bilinear_interp(x_grid, t_grid, values, x_query, t_query):    xi = np.clip(np.searchsorted(x_grid, x_query) - 1, 0, len(x_grid) - 2)    ti = np.clip(np.searchsorted(t_grid, t_query) - 1, 0, len(t_grid) - 2)    x0v, x1v = x_grid[xi], x_grid[xi + 1]    t0v, t1v = t_grid[ti], t_grid[ti + 1]    wx = (x_query - x0v) / (x1v - x0v + 1e-12)    wt = (t_query - t0v) / (t1v - t0v + 1e-12)    v00 = values[ti, xi]; v01 = values[ti, xi + 1]; v10 = values[ti + 1, xi]; v11 = values[ti + 1, xi + 1]    v0 = v00 * (1 - wx) + v01 * wx    v1 = v10 * (1 - wx) + v11 * wx    return v0 * (1 - wt) + v1 * wt@lru_cache(maxsize=8)def _burgers_fd_reference(bench, nx=201, nt_min=400):    """Explicit FD reference solution for Burgers' (no closed form exists)."""    x0, x1 = bench.x0, bench.x1    t1 = bench.t1    nu = bench.nu    x = np.linspace(x0, x1, nx)    dx = x[1] - x[0]    u = bench.initial_condition(x).astype(np.float64)    u[0] = 0.0; u[-1] = 0.0    dt_diff = dx * dx / (2.0 * nu + 1e-8)    dt_conv = dx / (np.max(np.abs(u)) + 1e-6)    dt = 0.4 * min(dt_diff, dt_conv)    nt = max(int(nt_min), int(np.ceil(t1 / dt)) + 1)    dt = t1 / nt    history = np.zeros((nt + 1, nx), dtype=np.float64)    history[0] = u    for n in range(1, nt + 1):        u_x = np.zeros_like(u); u_x[1:-1] = (u[2:] - u[:-2]) / (2 * dx)        u_xx = np.zeros_like(u); u_xx[1:-1] = (u[2:] - 2 * u[1:-1] + u[:-2]) / dx ** 2        u_new = u.copy()        u_new[1:-1] = u[1:-1] + dt * (-u[1:-1] * u_x[1:-1] + nu * u_xx[1:-1])        u_new[0] = 0.0; u_new[-1] = 0.0        u = u_new        history[n] = u    t_grid = np.linspace(0.0, t1, nt + 1)    return x, t_grid, historydef train_pinn_ode(cfg, bench):    device = torch.device(cfg.device if cfg.device == "cuda" and torch.cuda.is_available() else "cpu")    model = MLP(1, 1, cfg.hidden_layers, cfg.hidden_width, cfg.activation).to(device)    opt, use_lbfgs = _make_optimizer(model, cfg)    rng = np.random.default_rng(cfg.seed + 123)    t_col_np = rng.uniform(cfg.t0, cfg.t1, size=(int(cfg.n_collocation), 1)).astype(np.float32)    t_col = torch.tensor(t_col_np, device=device, requires_grad=True)    t0_tensor = torch.tensor([[cfg.t0]], device=device, requires_grad=True)    y0_target = torch.tensor([[1.0]], device=device)    def compute_loss():        y = model(t_col)        dy_dt = torch.autograd.grad(y, t_col, grad_outputs=torch.ones_like(y), create_graph=True, retain_graph=True)[0]        r = bench.residual(t_col, y, dy_dt)        loss_phys = torch.mean(r**2)        y0_pred = model(t0_tensor)        loss_ic = torch.mean((y0_pred - y0_target) ** 2)        return float(cfg.w_phys) * loss_phys + float(cfg.w_ic) * loss_ic    last_loss = _run_optimization(opt, use_lbfgs, compute_loss, cfg.n_steps)    t_eval = np.linspace(cfg.t0, cfg.t1, int(cfg.n_eval), dtype=np.float32).reshape(-1, 1)    y_true = bench.y_true(t_eval).reshape(-1, 1).astype(np.float32)    with torch.no_grad():        y_pred = model(torch.tensor(t_eval, device=device)).detach().cpu().numpy()    err = y_pred - y_true    mse = float(np.mean(err**2)); linf = float(np.max(np.abs(err)))    rel_l2 = float(np.linalg.norm(err) / (np.linalg.norm(y_true) + 1e-12))    return {"train_last_loss": last_loss, "val_mse": mse, "val_linf": linf, "val_rel_l2": rel_l2}def train_pinn_heat(cfg, bench):    device = torch.device(cfg.device if cfg.device == "cuda" and torch.cuda.is_available() else "cpu")    model = MLP(2, 1, cfg.hidden_layers, cfg.hidden_width, cfg.activation).to(device)    opt, use_lbfgs = _make_optimizer(model, cfg)    (x0, x1), (t0, t1) = bench.domain()    n_col = int(cfg.n_collocation)    n_bc = max(16, n_col // 8)    rng = np.random.default_rng(cfg.seed + 123)    x_col = torch.tensor(rng.uniform(x0, x1, size=(n_col, 1)).astype(np.float32), device=device, requires_grad=True)    t_col = torch.tensor(rng.uniform(t0, t1, size=(n_col, 1)).astype(np.float32), device=device, requires_grad=True)    x_ic_np = rng.uniform(x0, x1, size=(n_bc, 1)).astype(np.float32)    x_ic = torch.tensor(x_ic_np, device=device)    t_ic = torch.zeros_like(x_ic)    ic_target = torch.tensor(bench.initial_condition(x_ic_np).astype(np.float32), device=device)    t_bc_np = rng.uniform(t0, t1, size=(n_bc, 1)).astype(np.float32)    t_bc = torch.tensor(t_bc_np, device=device)    x_bc0 = torch.full_like(t_bc, x0); x_bc1 = torch.full_like(t_bc, x1)    bc0_np, bc1_np = bench.boundary_conditions(t_bc_np)    bc0_target = torch.tensor(np.asarray(bc0_np, dtype=np.float32), device=device)    bc1_target = torch.tensor(np.asarray(bc1_np, dtype=np.float32), device=device)    def compute_loss():        u = model(torch.cat([x_col, t_col], dim=1))        u_x = torch.autograd.grad(u, x_col, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]        u_xx = torch.autograd.grad(u_x, x_col, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0]        u_t = torch.autograd.grad(u, t_col, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]        res = bench.residual(x_col, t_col, u, u_t, u_xx)        loss_phys = torch.mean(res ** 2)        u_ic_pred = model(torch.cat([x_ic, t_ic], dim=1))        loss_ic = torch.mean((u_ic_pred - ic_target) ** 2)        u_bc0_pred = model(torch.cat([x_bc0, t_bc], dim=1))        u_bc1_pred = model(torch.cat([x_bc1, t_bc], dim=1))        loss_bc = torch.mean((u_bc0_pred - bc0_target) ** 2) + torch.mean((u_bc1_pred - bc1_target) ** 2)        return float(cfg.w_phys) * loss_phys + float(cfg.w_ic) * (loss_ic + loss_bc)    last_loss = _run_optimization(opt, use_lbfgs, compute_loss, cfg.n_steps)    n_eval = int(cfg.n_eval)    x_eval = np.linspace(x0, x1, n_eval, dtype=np.float32)    t_eval = np.linspace(t0, t1, n_eval, dtype=np.float32)    Xg, Tg = np.meshgrid(x_eval, t_eval)    x_flat, t_flat = Xg.reshape(-1, 1), Tg.reshape(-1, 1)    u_true = bench.analytic_solution(x_flat, t_flat).astype(np.float32)    with torch.no_grad():        xt_eval = torch.tensor(np.concatenate([x_flat, t_flat], axis=1), device=device)        u_pred = model(xt_eval).cpu().numpy()    err = u_pred - u_true    mse = float(np.mean(err ** 2)); linf = float(np.max(np.abs(err)))    rel_l2 = float(np.linalg.norm(err) / (np.linalg.norm(u_true) + 1e-12))    return {"train_last_loss": last_loss, "val_mse": mse, "val_linf": linf, "val_rel_l2": rel_l2}def train_pinn_wave(cfg, bench):    device = torch.device(cfg.device if cfg.device == "cuda" and torch.cuda.is_available() else "cpu")    model = MLP(2, 1, cfg.hidden_layers, cfg.hidden_width, cfg.activation).to(device)    opt, use_lbfgs = _make_optimizer(model, cfg)    (x0, x1), (t0, t1) = bench.domain()    n_col = int(cfg.n_collocation)    n_bc = max(16, n_col // 8)    rng = np.random.default_rng(cfg.seed + 123)    x_col = torch.tensor(rng.uniform(x0, x1, size=(n_col, 1)).astype(np.float32), device=device, requires_grad=True)    t_col = torch.tensor(rng.uniform(t0, t1, size=(n_col, 1)).astype(np.float32), device=device, requires_grad=True)    x_ic_np = rng.uniform(x0, x1, size=(n_bc, 1)).astype(np.float32)    x_ic = torch.tensor(x_ic_np, device=device)    t_ic = torch.zeros_like(x_ic, requires_grad=True)    ic_pos_target = torch.tensor(bench.initial_condition_u(x_ic_np).astype(np.float32), device=device)    ic_vel_target = torch.tensor(bench.initial_condition_u_t(x_ic_np).astype(np.float32), device=device)    t_bc_np = rng.uniform(t0, t1, size=(n_bc, 1)).astype(np.float32)    t_bc = torch.tensor(t_bc_np, device=device)    x_bc0 = torch.full_like(t_bc, x0); x_bc1 = torch.full_like(t_bc, x1)    bc0_np, bc1_np = bench.boundary_conditions(t_bc_np)    bc0_target = torch.tensor(np.asarray(bc0_np, dtype=np.float32), device=device)    bc1_target = torch.tensor(np.asarray(bc1_np, dtype=np.float32), device=device)    def compute_loss():        u = model(torch.cat([x_col, t_col], dim=1))        u_x = torch.autograd.grad(u, x_col, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]        u_xx = torch.autograd.grad(u_x, x_col, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0]        u_t = torch.autograd.grad(u, t_col, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]        u_tt = torch.autograd.grad(u_t, t_col, grad_outputs=torch.ones_like(u_t), create_graph=True, retain_graph=True)[0]        res = bench.residual(x_col, t_col, u, u_tt, u_xx)        loss_phys = torch.mean(res ** 2)        u_ic = model(torch.cat([x_ic, t_ic], dim=1))        u_ic_t = torch.autograd.grad(u_ic, t_ic, grad_outputs=torch.ones_like(u_ic), create_graph=True, retain_graph=True)[0]        loss_ic_pos = torch.mean((u_ic - ic_pos_target) ** 2)        loss_ic_vel = torch.mean((u_ic_t - ic_vel_target) ** 2)        u_bc0 = model(torch.cat([x_bc0, t_bc], dim=1))        u_bc1 = model(torch.cat([x_bc1, t_bc], dim=1))        loss_bc = torch.mean((u_bc0 - bc0_target) ** 2) + torch.mean((u_bc1 - bc1_target) ** 2)        return float(cfg.w_phys) * loss_phys + float(cfg.w_ic) * (loss_ic_pos + loss_ic_vel + loss_bc)    last_loss = _run_optimization(opt, use_lbfgs, compute_loss, cfg.n_steps)    n_eval = int(cfg.n_eval)    x_eval = np.linspace(x0, x1, n_eval, dtype=np.float32)    t_eval = np.linspace(t0, t1, n_eval, dtype=np.float32)    Xg, Tg = np.meshgrid(x_eval, t_eval)    x_flat, t_flat = Xg.reshape(-1, 1), Tg.reshape(-1, 1)    u_true = bench.analytic_solution(x_flat, t_flat).astype(np.float32)    with torch.no_grad():        xt_eval = torch.tensor(np.concatenate([x_flat, t_flat], axis=1), device=device)        u_pred = model(xt_eval).cpu().numpy()    err = u_pred - u_true    mse = float(np.mean(err ** 2)); linf = float(np.max(np.abs(err)))    rel_l2 = float(np.linalg.norm(err) / (np.linalg.norm(u_true) + 1e-12))    return {"train_last_loss": last_loss, "val_mse": mse, "val_linf": linf, "val_rel_l2": rel_l2}def train_pinn_burgers(cfg, bench):    device = torch.device(cfg.device if cfg.device == "cuda" and torch.cuda.is_available() else "cpu")    model = MLP(2, 1, cfg.hidden_layers, cfg.hidden_width, cfg.activation).to(device)    opt, use_lbfgs = _make_optimizer(model, cfg)    (x0, x1), (t0, t1) = bench.domain()    n_col = int(cfg.n_collocation)    n_bc = max(16, n_col // 8)    rng = np.random.default_rng(cfg.seed + 123)    x_col = torch.tensor(rng.uniform(x0, x1, size=(n_col, 1)).astype(np.float32), device=device, requires_grad=True)    t_col = torch.tensor(rng.uniform(t0, t1, size=(n_col, 1)).astype(np.float32), device=device, requires_grad=True)    x_ic_np = rng.uniform(x0, x1, size=(n_bc, 1)).astype(np.float32)    x_ic = torch.tensor(x_ic_np, device=device)    t_ic = torch.zeros_like(x_ic)    ic_target = torch.tensor(bench.initial_condition(x_ic_np).astype(np.float32), device=device)    t_bc_np = rng.uniform(t0, t1, size=(n_bc, 1)).astype(np.float32)    t_bc = torch.tensor(t_bc_np, device=device)    x_bc0 = torch.full_like(t_bc, x0); x_bc1 = torch.full_like(t_bc, x1)    bc0_np, bc1_np = bench.boundary_conditions(t_bc_np)    bc0_target = torch.tensor(np.asarray(bc0_np, dtype=np.float32), device=device)    bc1_target = torch.tensor(np.asarray(bc1_np, dtype=np.float32), device=device)    def compute_loss():        u = model(torch.cat([x_col, t_col], dim=1))        u_x = torch.autograd.grad(u, x_col, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]        u_xx = torch.autograd.grad(u_x, x_col, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0]        u_t = torch.autograd.grad(u, t_col, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]        res = bench.residual(x_col, t_col, u, u_t, u_x, u_xx)        loss_phys = torch.mean(res ** 2)        u_ic_pred = model(torch.cat([x_ic, t_ic], dim=1))        loss_ic = torch.mean((u_ic_pred - ic_target) ** 2)        u_bc0_pred = model(torch.cat([x_bc0, t_bc], dim=1))        u_bc1_pred = model(torch.cat([x_bc1, t_bc], dim=1))        loss_bc = torch.mean((u_bc0_pred - bc0_target) ** 2) + torch.mean((u_bc1_pred - bc1_target) ** 2)        return float(cfg.w_phys) * loss_phys + float(cfg.w_ic) * (loss_ic + loss_bc)    last_loss = _run_optimization(opt, use_lbfgs, compute_loss, cfg.n_steps)    x_grid, t_grid, history = _burgers_fd_reference(bench)    n_eval = int(cfg.n_eval)    x_eval = np.linspace(x0, x1, n_eval, dtype=np.float64)    t_eval = np.linspace(t0, t1, n_eval, dtype=np.float64)    Xg, Tg = np.meshgrid(x_eval, t_eval)    x_flat, t_flat = Xg.reshape(-1), Tg.reshape(-1)    u_true = _bilinear_interp(x_grid, t_grid, history, x_flat, t_flat).astype(np.float32).reshape(-1, 1)    with torch.no_grad():        xt_eval = torch.tensor(np.stack([x_flat, t_flat], axis=1).astype(np.float32), device=device)        u_pred = model(xt_eval).cpu().numpy()    err = u_pred - u_true    mse = float(np.mean(err ** 2)); linf = float(np.max(np.abs(err)))    rel_l2 = float(np.linalg.norm(err) / (np.linalg.norm(u_true) + 1e-12))    return {"train_last_loss": last_loss, "val_mse": mse, "val_linf": linf, "val_rel_l2": rel_l2}def train_pinn(cfg: TrainConfig) -> dict:    """Real PINN trainer dispatch - matches src/training/pinn_trainer.py exactly."""    set_seed(cfg.seed)    bench = get_benchmark(cfg.benchmark_type)    if cfg.benchmark_type == "ode":        metrics = train_pinn_ode(cfg, bench)    elif cfg.benchmark_type == "heat":        metrics = train_pinn_heat(cfg, bench)    elif cfg.benchmark_type == "burgers":        metrics = train_pinn_burgers(cfg, bench)    elif cfg.benchmark_type == "wave":        metrics = train_pinn_wave(cfg, bench)    else:        raise ValueError(f"No real trainer wired up for benchmark_type={cfg.benchmark_type!r}")    return {"config": asdict(cfg), **metrics}print("Real PINN trainers ready: ODE, Heat, Burgers, Wave (all use torch.autograd physics residuals).")

In [ ]:
def _ga_numpy(fitness_func, lb, ub, sol_per_pop=10, n_generations=8,             num_parents_mating=4, mutation_rate=0.2, seed=0):    rng = np.random.default_rng(seed)    dim = len(lb)    pop = lb + (ub - lb) * rng.random(size=(sol_per_pop, dim))    fitnesses = np.array([fitness_func(ind) for ind in pop], dtype=float)    best_idx = np.argmax(fitnesses)    best_ind = pop[best_idx].copy()    best_fit = float(fitnesses[best_idx])    history = [-best_fit]    diversity_history = [{"generation": 0, "diversity": compute_population_diversity(pop, lb, ub)}]    for gen in range(1, int(n_generations) + 1):        parents = []        for _ in range(num_parents_mating):            tourn_idx = rng.choice(sol_per_pop, size=3, replace=False)            winner = tourn_idx[np.argmax(fitnesses[tourn_idx])]            parents.append(pop[winner])        parents = np.array(parents)        next_pop = [best_ind.copy()]        while len(next_pop) < sol_per_pop:            p1_idx, p2_idx = rng.choice(len(parents), size=2, replace=False)            cross_pt = rng.integers(1, dim)            child = np.concatenate([parents[p1_idx][:cross_pt], parents[p2_idx][cross_pt:]])            for d in range(dim):                if rng.random() < mutation_rate:                    child[d] = lb[d] + (ub[d] - lb[d]) * rng.random()            child = np.clip(child, lb, ub)            next_pop.append(child)        pop = np.array(next_pop)        fitnesses = np.array([fitness_func(ind) for ind in pop], dtype=float)        best_idx = np.argmax(fitnesses)        if fitnesses[best_idx] > best_fit:            best_fit = float(fitnesses[best_idx])            best_ind = pop[best_idx].copy()        history.append(-best_fit)        diversity_history.append({"generation": gen, "diversity": compute_population_diversity(pop, lb, ub)})    return best_ind, best_fit, history, diversity_historydef run_ga(out_dir, benchmark_type="ode", seed=0, n_generations=8, sol_per_pop=10,           num_parents_mating=4, n_steps=1200):    space = SearchSpace()    base = TrainConfig(seed=seed, n_steps=n_steps, benchmark_type=benchmark_type, device=DEVICE)    lb, ub = space.get_bounds()    def fitness_func(solution):        cfg = decode_solution(np.asarray(solution), space, base)        return -float(train_pinn(cfg)["val_rel_l2"])    best_ind, best_fit, history, diversity_history = _ga_numpy(        fitness_func, lb, ub, sol_per_pop=sol_per_pop, n_generations=n_generations,        num_parents_mating=num_parents_mating, seed=seed)    best_cfg = decode_solution(best_ind, space, base)    best_metrics = train_pinn(best_cfg)    best_metrics["history"] = history    best_metrics["diversity_history"] = diversity_history    best_metrics["optimizer_name"] = "GA"    ensure_dir(out_dir)    save_json(f"{out_dir}/ga_best_metrics.json", best_metrics)    return best_metricsprint("GA ready.")

In [ ]:
def _pso_numpy(func, lb, ub, swarmsize=12, maxiter=8, w=0.7, c1=1.5, c2=1.5, seed=0):    rng = np.random.default_rng(seed)    dim = len(lb)    v_max = (ub - lb) * 0.2    X = lb + (ub - lb) * rng.random(size=(swarmsize, dim))    V = -v_max + 2 * v_max * rng.random(size=(swarmsize, dim))    P = X.copy()    P_fit = np.array([func(x) for x in X], dtype=float)    best_idx = np.argmin(P_fit)    gbest = P[best_idx].copy()    gbest_fit = float(P_fit[best_idx])    history = [gbest_fit]    diversity_history = [{"iteration": 0, "diversity": compute_population_diversity(X, lb, ub)}]    for it in range(1, int(maxiter) + 1):        r1 = rng.random(size=(swarmsize, dim))        r2 = rng.random(size=(swarmsize, dim))        V = w * V + c1 * r1 * (P - X) + c2 * r2 * (gbest - X)        V = np.clip(V, -v_max, v_max)        X = np.clip(X + V, lb, ub)        for i in range(swarmsize):            fit = func(X[i])            if fit < P_fit[i]:                P_fit[i] = fit                P[i] = X[i].copy()                if fit < gbest_fit:                    gbest_fit = float(fit)                    gbest = X[i].copy()        history.append(gbest_fit)        diversity_history.append({"iteration": it, "diversity": compute_population_diversity(X, lb, ub)})    return gbest, gbest_fit, history, diversity_historydef run_pso(out_dir, benchmark_type="ode", seed=0, swarmsize=12, maxiter=8, n_steps=1200):    space = SearchSpace()    base = TrainConfig(seed=seed, n_steps=n_steps, benchmark_type=benchmark_type, device=DEVICE)    lb, ub = space.get_bounds()    def objective(x):        cfg = decode_solution(np.asarray(x), space, base)        return float(train_pinn(cfg)["val_rel_l2"])    best_x, best_f, history, diversity_history = _pso_numpy(        objective, lb, ub, swarmsize=swarmsize, maxiter=maxiter, seed=seed)    best_cfg = decode_solution(best_x, space, base)    best_metrics = train_pinn(best_cfg)    best_metrics["history"] = history    best_metrics["diversity_history"] = diversity_history    best_metrics["optimizer_name"] = "PSO"    ensure_dir(out_dir)    save_json(f"{out_dir}/pso_best_metrics.json", best_metrics)    return best_metricsprint("PSO ready.")

In [ ]:
def run_aco(out_dir, benchmark_type="ode", seed=0, n_ants=10, n_iterations=8, n_steps=1200):    rng = np.random.default_rng(seed)    space = SearchSpace()    base = TrainConfig(seed=seed, n_steps=n_steps, benchmark_type=benchmark_type, device=DEVICE)    lb, ub = space.get_bounds()    dim = len(lb)    def objective(x):        cfg = decode_solution(x, space, base)        return float(train_pinn(cfg)["val_rel_l2"])    archive_size = max(10, n_ants)    A = lb + (ub - lb) * rng.random(size=(archive_size, dim))    f = np.array([objective(x) for x in A], dtype=float)    history = [float(np.min(f))]    diversity_history = [{"iteration": 0, "diversity": compute_population_diversity(A, lb, ub)}]    for it in range(1, int(n_iterations) + 1):        order = np.argsort(f)        A = A[order]; f = f[order]        k_idx = np.arange(archive_size)        w = (1.0 / (0.5 * archive_size * np.sqrt(2.0 * np.pi))) * np.exp(-(k_idx ** 2) / (2.0 * (0.5 * archive_size) ** 2))        w = w / np.sum(w)        sigma = np.zeros(dim, dtype=float)        for d in range(dim):            diff = np.abs(A[:, d] - np.dot(w, A[:, d]))            sigma[d] = 0.85 * np.mean(diff) + 1e-8        new_X = np.zeros((n_ants, dim), dtype=float)        new_f = np.zeros(n_ants, dtype=float)        for i in range(n_ants):            x_new = np.zeros(dim, dtype=float)            for d in range(dim):                idx = rng.choice(archive_size, p=w)                val = rng.normal(loc=A[idx, d], scale=sigma[d])                x_new[d] = np.clip(val, lb[d], ub[d])            new_X[i] = x_new            new_f[i] = objective(x_new)        A = np.vstack([A, new_X]); f = np.concatenate([f, new_f])        order = np.argsort(f)        A = A[order][:archive_size]; f = f[order][:archive_size]        history.append(float(f[0]))        diversity_history.append({"iteration": it, "diversity": compute_population_diversity(A, lb, ub)})    best_cfg = decode_solution(A[0], space, base)    best_metrics = train_pinn(best_cfg)    best_metrics["history"] = history    best_metrics["diversity_history"] = diversity_history    best_metrics["optimizer_name"] = "ACO"    ensure_dir(out_dir)    save_json(f"{out_dir}/aco_best_metrics.json", best_metrics)    return best_metricsprint("ACO ready.")

In [ ]:
def run_fuzzy_ga(out_dir, benchmark_type="ode", seed=0, n_generations=8,              sol_per_pop=10, num_parents_mating=4, n_steps=1200):    rng = np.random.default_rng(seed)    space = SearchSpace()    base = TrainConfig(seed=seed, n_steps=n_steps, benchmark_type=benchmark_type, device=DEVICE)    lb, ub = space.get_bounds()    dim = len(lb)    flc = FuzzyController()    def objective(x):        cfg = decode_solution(x, space, base)        return float(train_pinn(cfg)["val_rel_l2"])    pop = lb + (ub - lb) * rng.random(size=(sol_per_pop, dim))    fitnesses = np.array([objective(ind) for ind in pop], dtype=float)    best_idx = np.argmin(fitnesses)    best_ind = pop[best_idx].copy()    best_fit = float(fitnesses[best_idx])    history = [best_fit]    prev_best_fit = best_fit    fuzzy_adaptations = []    for gen in range(1, int(n_generations) + 1):        diversity = compute_population_diversity(pop, lb, ub)        improvement = float(max(0.0, (prev_best_fit - best_fit) / (prev_best_fit + 1e-12)))        progress = float(gen / n_generations)        explore_w, exploit_w = flc.evaluate(diversity, improvement, progress)        mutation_rate = float(np.clip(0.05 + 0.35 * explore_w, 0.05, 0.45))        crossover_prob = float(np.clip(0.50 + 0.45 * exploit_w, 0.50, 0.95))        fuzzy_adaptations.append({"generation": gen, "diversity": diversity,                                 "mutation_rate": mutation_rate, "crossover_prob": crossover_prob})        prev_best_fit = best_fit        parents = []        for _ in range(num_parents_mating):            tourn_idx = rng.choice(sol_per_pop, size=3, replace=False)            winner = tourn_idx[np.argmin(fitnesses[tourn_idx])]            parents.append(pop[winner])        parents = np.array(parents)        next_pop = [best_ind.copy()]        while len(next_pop) < sol_per_pop:            p1_idx, p2_idx = rng.choice(len(parents), size=2, replace=False)            p1, p2 = parents[p1_idx], parents[p2_idx]            if rng.random() < crossover_prob:                cross_pt = rng.integers(1, dim)                child = np.concatenate([p1[:cross_pt], p2[cross_pt:]])            else:                child = p1.copy()            for d in range(dim):                if rng.random() < mutation_rate:                    step = (ub[d] - lb[d]) * (0.1 + 0.4 * explore_w)                    child[d] += rng.normal(0.0, step)            child = np.clip(child, lb, ub)            next_pop.append(child)        pop = np.array(next_pop)        fitnesses = np.array([objective(ind) for ind in pop], dtype=float)        best_idx = np.argmin(fitnesses)        if fitnesses[best_idx] < best_fit:            best_fit = float(fitnesses[best_idx])            best_ind = pop[best_idx].copy()        history.append(best_fit)    best_cfg = decode_solution(best_ind, space, base)    best_metrics = train_pinn(best_cfg)    best_metrics["history"] = history    best_metrics["fuzzy_adaptations"] = fuzzy_adaptations    best_metrics["diversity_history"] = [{"generation": a["generation"], "diversity": a["diversity"]} for a in fuzzy_adaptations]    best_metrics["optimizer_name"] = "Fuzzy-GA"    ensure_dir(out_dir)    save_json(f"{out_dir}/fuzzy_ga_best_metrics.json", best_metrics)    return best_metricsdef run_fuzzy_pso(out_dir, benchmark_type="ode", seed=0, swarmsize=12, maxiter=8, n_steps=1200):    rng = np.random.default_rng(seed)    space = SearchSpace()    base = TrainConfig(seed=seed, n_steps=n_steps, benchmark_type=benchmark_type, device=DEVICE)    lb, ub = space.get_bounds()    dim = len(lb)    v_max = (ub - lb) * 0.25    flc = FuzzyController()    def objective(x):        cfg = decode_solution(x, space, base)        return float(train_pinn(cfg)["val_rel_l2"])    X = lb + (ub - lb) * rng.random(size=(swarmsize, dim))    V = -v_max + 2 * v_max * rng.random(size=(swarmsize, dim))    P = X.copy()    P_fit = np.array([objective(x) for x in X], dtype=float)    best_idx = np.argmin(P_fit)    gbest = P[best_idx].copy()    gbest_fit = float(P_fit[best_idx])    history = [gbest_fit]    prev_gbest_fit = gbest_fit    fuzzy_adaptations = []    for it in range(1, int(maxiter) + 1):        diversity = compute_population_diversity(X, lb, ub)        improvement = float(max(0.0, (prev_gbest_fit - gbest_fit) / (prev_gbest_fit + 1e-12)))        progress = float(it / maxiter)        explore_w, exploit_w = flc.evaluate(diversity, improvement, progress)        w = 0.3 + 0.6 * explore_w        c2 = 1.0 + 1.5 * exploit_w        c1 = float(np.clip(3.2 - c2, 0.8, 2.5))        fuzzy_adaptations.append({"iteration": it, "diversity": diversity, "w": w, "c1": c1, "c2": c2})        prev_gbest_fit = gbest_fit        r1 = rng.random(size=(swarmsize, dim)); r2 = rng.random(size=(swarmsize, dim))        V = w * V + c1 * r1 * (P - X) + c2 * r2 * (gbest - X)        V = np.clip(V, -v_max, v_max)        X = np.clip(X + V, lb, ub)        for i in range(swarmsize):            fit = objective(X[i])            if fit < P_fit[i]:                P_fit[i] = fit                P[i] = X[i].copy()                if fit < gbest_fit:                    gbest_fit = float(fit)                    gbest = X[i].copy()        history.append(gbest_fit)    best_cfg = decode_solution(gbest, space, base)    best_metrics = train_pinn(best_cfg)    best_metrics["history"] = history    best_metrics["fuzzy_adaptations"] = fuzzy_adaptations    best_metrics["diversity_history"] = [{"iteration": a["iteration"], "diversity": a["diversity"]} for a in fuzzy_adaptations]    best_metrics["optimizer_name"] = "Fuzzy-PSO"    ensure_dir(out_dir)    save_json(f"{out_dir}/fuzzy_pso_best_metrics.json", best_metrics)    return best_metricsdef run_fuzzy_aco(out_dir, benchmark_type="ode", seed=0, n_ants=10, n_iterations=8, n_steps=1200):    rng = np.random.default_rng(seed)    space = SearchSpace()    base = TrainConfig(seed=seed, n_steps=n_steps, benchmark_type=benchmark_type, device=DEVICE)    lb, ub = space.get_bounds()    dim = len(lb)    flc = FuzzyController()    def objective(x):        cfg = decode_solution(x, space, base)        return float(train_pinn(cfg)["val_rel_l2"])    archive_size = max(10, n_ants)    A = lb + (ub - lb) * rng.random(size=(archive_size, dim))    f = np.array([objective(x) for x in A], dtype=float)    best_f = float(np.min(f))    history = [best_f]    prev_best_f = best_f    fuzzy_adaptations = []    for it in range(1, int(n_iterations) + 1):        diversity = compute_population_diversity(A, lb, ub)        improvement = float(max(0.0, (prev_best_f - best_f) / (prev_best_f + 1e-12)))        progress = float(it / n_iterations)        explore_w, exploit_w = flc.evaluate(diversity, improvement, progress)        zeta = float(np.clip(0.35 + 0.85 * explore_w, 0.3, 1.2))        q = float(np.clip(0.15 + 0.65 * (1.0 - exploit_w), 0.1, 0.9))        fuzzy_adaptations.append({"iteration": it, "diversity": diversity, "zeta": zeta, "q": q})        prev_best_f = best_f        order = np.argsort(f)        A = A[order]; f = f[order]        k_idx = np.arange(archive_size)        w_arch = (1.0 / (q * archive_size * np.sqrt(2.0 * np.pi))) * np.exp(-(k_idx ** 2) / (2.0 * (q * archive_size) ** 2))        w_arch = w_arch / np.sum(w_arch)        sigma = np.zeros(dim, dtype=float)        for d in range(dim):            diff = np.abs(A[:, d] - np.dot(w_arch, A[:, d]))            sigma[d] = zeta * np.mean(diff) + 1e-8        new_X = np.zeros((n_ants, dim), dtype=float)        new_f = np.zeros(n_ants, dtype=float)        for i in range(n_ants):            x_new = np.zeros(dim, dtype=float)            for d in range(dim):                idx = rng.choice(archive_size, p=w_arch)                val = rng.normal(loc=A[idx, d], scale=sigma[d])                x_new[d] = np.clip(val, lb[d], ub[d])            new_X[i] = x_new            new_f[i] = objective(x_new)        A = np.vstack([A, new_X]); f = np.concatenate([f, new_f])        order = np.argsort(f)        A = A[order][:archive_size]; f = f[order][:archive_size]        best_f = float(f[0])        history.append(best_f)    best_cfg = decode_solution(A[0], space, base)    best_metrics = train_pinn(best_cfg)    best_metrics["history"] = history    best_metrics["fuzzy_adaptations"] = fuzzy_adaptations    best_metrics["diversity_history"] = [{"iteration": a["iteration"], "diversity": a["diversity"]} for a in fuzzy_adaptations]    best_metrics["optimizer_name"] = "Fuzzy-ACO"    ensure_dir(out_dir)    save_json(f"{out_dir}/fuzzy_aco_best_metrics.json", best_metrics)    return best_metricsprint("Fuzzy-GA, Fuzzy-PSO, Fuzzy-ACO ready.")

In [ ]:
# ============================================================# CONFIGURATION -- run with QUICK_MODE=True first to smoke-test# ============================================================QUICK_MODE = True  # <-- set to False for the full multi-hour real runALGORITHMS = ["GA", "PSO", "ACO", "Fuzzy-GA", "Fuzzy-PSO", "Fuzzy-ACO"]BENCHMARKS = ["ode", "heat", "burgers", "wave"]SEEDS = [0, 1, 2]if QUICK_MODE:    # Smoke-test settings: tiny population/generations and n_steps=1 gradient step,    # just to validate the full pipeline runs end-to-end in a couple of minutes.    N_STEPS = 1    N_GENERATIONS_GA, SOL_PER_POP_GA = 4, 6    SWARMSIZE_PSO, MAXITER_PSO = 6, 4    N_ANTS_ACO, N_ITERATIONS_ACO = 6, 4else:    # Real full-scale settings (matches src/hpo/comparison.py non-quick defaults).    N_STEPS = 1200    N_GENERATIONS_GA, SOL_PER_POP_GA = 8, 10    SWARMSIZE_PSO, MAXITER_PSO = 12, 8    N_ANTS_ACO, N_ITERATIONS_ACO = 10, 8algo_funcs = {    "GA": run_ga, "PSO": run_pso, "ACO": run_aco,    "Fuzzy-GA": run_fuzzy_ga, "Fuzzy-PSO": run_fuzzy_pso, "Fuzzy-ACO": run_fuzzy_aco,}tasks = [(a, b, s) for a in ALGORITHMS for b in BENCHMARKS for s in SEEDS]print(f"Mode: {'QUICK smoke-test' if QUICK_MODE else 'FULL real run'}")print(f"Total optimization tasks: {len(tasks)} (6 algorithms x 4 benchmarks x 3 seeds)")print(f"n_steps per PINN training call: {N_STEPS}")if not QUICK_MODE:    approx_evals_per_task = SOL_PER_POP_GA * (N_GENERATIONS_GA + 1)  # rough order-of-magnitude    print(f"WARNING: full run performs real gradient descent -- approx {approx_evals_per_task}+ "          f"PINN training calls x {N_STEPS} steps per task x {len(tasks)} tasks. "          f"Expect several hours on a T4 GPU. Ensure Runtime -> GPU is enabled.")

In [ ]:
results = {}start_time = time.time()pbar = tqdm(total=len(tasks), desc="Optimization Tasks")for algo, bench, seed in tasks:    out_subdir = f"{OUTPUT_DIR}/{algo}_{bench}_{seed}"    ensure_dir(out_subdir)    try:        func = algo_funcs[algo]        if algo in ("GA", "Fuzzy-GA"):            result = func(out_subdir, benchmark_type=bench, seed=seed,                         n_generations=N_GENERATIONS_GA, sol_per_pop=SOL_PER_POP_GA, n_steps=N_STEPS)        elif algo in ("PSO", "Fuzzy-PSO"):            result = func(out_subdir, benchmark_type=bench, seed=seed,                         swarmsize=SWARMSIZE_PSO, maxiter=MAXITER_PSO, n_steps=N_STEPS)        elif algo in ("ACO", "Fuzzy-ACO"):            result = func(out_subdir, benchmark_type=bench, seed=seed,                         n_ants=N_ANTS_ACO, n_iterations=N_ITERATIONS_ACO, n_steps=N_STEPS)        results[f"{algo}_{bench}_{seed}"] = result    except Exception as e:        print(f"Error on {algo}/{bench}/{seed}: {e}")    pbar.update(1)pbar.close()elapsed = time.time() - start_timeprint(f"\nCompleted {len(results)}/{len(tasks)} tasks in {elapsed/60:.1f} min")

In [ ]:
comparison_results = {}for algo in ALGORITHMS:    comparison_results[algo] = {}    for bench in BENCHMARKS:        vals, divs = [], []        for seed in SEEDS:            key = f"{algo}_{bench}_{seed}"            if key in results:                vals.append(results[key].get("val_rel_l2", float('nan')))                dh = results[key].get("diversity_history", [])                if dh:                    divs.append(np.mean([d["diversity"] for d in dh]))        if vals:            comparison_results[algo][bench] = {                "mean_l2": float(np.mean(vals)), "std_l2": float(np.std(vals)),                "min_l2": float(np.min(vals)), "max_l2": float(np.max(vals)),                "mean_diversity": float(np.mean(divs)) if divs else 0.0,            }save_json(f"{OUTPUT_DIR}/hpo_comparison_results.json", comparison_results)print("=" * 90)print(f"RESULTS SUMMARY ({'QUICK smoke-test' if QUICK_MODE else 'FULL real run'})")print("=" * 90)for algo in ALGORITHMS:    print(f"\n{algo}:")    for bench in BENCHMARKS:        if bench in comparison_results.get(algo, {}):            r = comparison_results[algo][bench]            print(f"  {bench:10s}: L2={r['mean_l2']:.4f}+/-{r['std_l2']:.4f}  Div={r['mean_diversity']:.3f}")print("=" * 90)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))for idx, bench in enumerate(BENCHMARKS):    ax = axes[idx // 2, idx % 2]    for algo in ALGORITHMS:        vals = [results[f"{algo}_{bench}_{s}"]["history"] for s in SEEDS if f"{algo}_{bench}_{s}" in results]        if vals:            ax.plot(np.mean([np.array(v) for v in vals], axis=0), label=algo, linewidth=2)    ax.set_xlabel("Iteration"); ax.set_ylabel("Validation L2 Error"); ax.set_title(bench.upper())    ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig(f"{OUTPUT_DIR}/convergence_comparison.png", dpi=150, bbox_inches="tight")plt.show()fig, axes = plt.subplots(2, 2, figsize=(14, 10))for idx, bench in enumerate(BENCHMARKS):    ax = axes[idx // 2, idx % 2]    for algo in ALGORITHMS:        divs = [[d["diversity"] for d in results[f"{algo}_{bench}_{s}"]["diversity_history"]]                for s in SEEDS if f"{algo}_{bench}_{s}" in results]        if divs:            ax.plot(np.mean([np.array(d) for d in divs], axis=0), label=algo, linewidth=2)    ax.set_xlabel("Iteration/Generation"); ax.set_ylabel("Population Diversity"); ax.set_title(bench.upper())    ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig(f"{OUTPUT_DIR}/diversity_trajectories.png", dpi=150, bbox_inches="tight")plt.show()fig, ax = plt.subplots(figsize=(12, 6))x_pos = np.arange(len(ALGORITHMS)); width = 0.2for idx, bench in enumerate(BENCHMARKS):    means = [comparison_results.get(a, {}).get(bench, {}).get("mean_l2", 0.0) for a in ALGORITHMS]    ax.bar(x_pos + idx * width, means, width, label=bench)ax.set_xlabel("Algorithm"); ax.set_ylabel("Mean Validation L2 Error")ax.set_title("Performance Comparison Across Benchmarks")ax.set_xticks(x_pos + width * 1.5); ax.set_xticklabels(ALGORITHMS, rotation=45)ax.legend(); ax.grid(True, alpha=0.3, axis="y")plt.tight_layout()plt.savefig(f"{OUTPUT_DIR}/performance_comparison.png", dpi=150, bbox_inches="tight")plt.show()

In [ ]:
report = []report.append("# NSYS 2026 Manuscript: HPO for Physics-Informed Neural Networks\n")report.append(f"Mode: {'QUICK smoke-test (not for publication)' if QUICK_MODE else 'FULL real run'}\n")report.append("## Experimental Setup\n")report.append("- 6 algorithms: GA, PSO, ACO, Fuzzy-GA, Fuzzy-PSO, Fuzzy-ACO")report.append("- 4 benchmarks: ODE, Heat, Burgers, Wave (real PyTorch PINN training, autograd physics residuals)")report.append(f"- 3 seeds: {SEEDS}")report.append(f"- n_steps per training call: {N_STEPS}\n")report.append("## Results Summary\n")report.append("| Algorithm | ODE | Heat | Burgers | Wave | Mean |")report.append("|-----------|-----|------|---------|------|------|")algo_means = {}for algo in ALGORITHMS:    vals = [comparison_results[algo][b]["mean_l2"] for b in BENCHMARKS if b in comparison_results.get(algo, {})]    if vals:        algo_means[algo] = np.mean(vals)        row = f"| {algo} |"        for b in BENCHMARKS:            row += f" {comparison_results[algo][b]['mean_l2']:.4f} |" if b in comparison_results[algo] else " - |"        row += f" {np.mean(vals):.4f} |"        report.append(row)report.append("\n## Diversity Analysis\n")report.append("| Algorithm | Mean Diversity |")report.append("|-----------|-----------------|")for algo in ALGORITHMS:    divs = [comparison_results[algo][b]["mean_diversity"] for b in BENCHMARKS if b in comparison_results.get(algo, {})]    if divs:        report.append(f"| {algo} | {np.mean(divs):.4f} |")if algo_means:    best_algo = min(algo_means, key=algo_means.get)    report.append(f"\n## Key Findings\n")    report.append(f"- Best overall algorithm: {best_algo} (mean L2 error: {algo_means[best_algo]:.4f})")    report.append(f"- With only {len(SEEDS)} seeds this is not statistically powered for significance testing; "                  f"rank differences should be read as descriptive.")report_text = "\n".join(report)with open(f"{OUTPUT_DIR}/MANUSCRIPT_REPORT.md", 'w') as f:    f.write(report_text)print(report_text)

In [ ]:
import shutilshutil.make_archive(f"{OUTPUT_DIR}/nsys2026_results", 'zip', OUTPUT_DIR)zip_path = f"{OUTPUT_DIR}/nsys2026_results.zip"print(f"Results packaged: {zip_path}")print(f"Size: {os.path.getsize(zip_path) / (1024**2):.1f} MB")try:    from google.colab import files    files.download(zip_path)except ImportError:    print("Not running in Colab -- download skipped. File is at:", zip_path)

## Notes- **QUICK_MODE**: Run once with `QUICK_MODE = True` (Configuration cell) to confirm  everything executes end-to-end (a few minutes). Then set `QUICK_MODE = False` and  **Runtime → Run all** again for the real multi-hour benchmark.- All four benchmark trainers (ODE, Heat, Burgers, Wave) use **real PyTorch autograd**  to compute PDE residuals and train an `MLP` — this mirrors `src/training/benchmark_factory.py`  exactly, no synthetic placeholder metrics.- Results are saved incrementally to `/content/nsys2026_results/` as each task finishes,  so a Colab disconnect part-way through does not lose completed tasks (re-running the  "Run all tasks" cell will simply overwrite/redo — no resume logic is included).- If Colab disconnects on a free-tier session during the full run, consider Colab Pro  for longer uninterrupted runtimes, or split `BENCHMARKS`/`ALGORITHMS` into smaller  chunks and run sequentially across sessions.